In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os, gc


In [3]:

src_path = os.path.abspath(os.path.join('..', 'Preprocessing'))

if src_path not in sys.path:
    sys.path.append(src_path)

from DataPreprocessor import DataPreprocessor
from DevicePreprocessor import DevicePreprocessor

In [4]:
data_folder_path = '../data/'
train_identity = pd.read_csv(f'{data_folder_path}train_identity.csv')
train_transaction = pd.read_csv(f'{data_folder_path}train_transaction.csv')
test_identity = pd.read_csv(f'{data_folder_path}test_identity.csv')
test_transaction = pd.read_csv(f'{data_folder_path}test_transaction.csv')

# Merge the data
train_data = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
test_data = pd.merge(test_transaction, test_identity, on='TransactionID', how='left')

del train_identity, train_transaction, test_identity, test_transaction
gc.collect()

0

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report
import xgboost as xgb
import joblib

y = train_data['isFraud']
X = train_data.drop(['isFraud', 'TransactionID'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False, random_state=42)

print("Assembling pipeline...")
scale_weight = (len(y_train) - sum(y_train)) / sum(y_train) # Dynamically calculate imbalance ratio

shopify_pipeline = Pipeline([
    ('device_preprocessor', DevicePreprocessor(min_frequency=250)),
    ('general_preprocessor', DataPreprocessor(nan_threshold=0.01)),
    ('autoencoder_bridge', AutoencoderPipeline(AutoencoderFeatureExtractor(epochs=35))),
    ('classifier', xgb.XGBClassifier(
        n_estimators=700,
        max_depth=6,
        learning_rate=0.026,
        min_child_weight=8,
        subsample=0.957,
        colsample_bytree= 0.689,
        scale_pos_weight=scale_weight,
        tree_method='hist',
        enable_categorical=True,
        random_state=42,
        eval_metric='aucpr', # Optimize for precision-recall area,
    ))
])

print("Training the pipeline (this may take a few minutes)...")
shopify_pipeline.fit(X_train, y_train)

# ==========================================
# 4. EVALUATION & EXPORT
# ==========================================
print("Evaluating model...")
# Predict probabilities (not just 0 or 1) for the PR curve
y_probs = shopify_pipeline.predict_proba(X_test)[:, 1]
y_preds = shopify_pipeline.predict(X_test)

# AUPRC is the gold standard for < 5% class imbalance
auprc = average_precision_score(y_test, y_probs)
roc_auc = roc_auc_score(y_test, y_probs)

print(f"\n--- Model Performance ---")
print(f"PR-AUC (Average Precision): {auprc:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_preds))

print("Saving production artifact...")
joblib.dump(shopify_pipeline, 'shopify_fraud_pipeline.joblib')
print("✅ Pipeline saved successfully as 'shopify_fraud_pipeline.joblib'")



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report

# Use only numeric columns
X_train_num = X_train.select_dtypes(include='number')
X_test_num  = X_test.select_dtypes(include='number')

# Median imputation — fit on train only
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_num)
X_test_imp  = imputer.transform(X_test_num)

lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

lr.fit(X_train_imp, y_train)

lr_probs = lr.predict_proba(X_test_imp)[:, 1]
lr_preds = lr.predict(X_test_imp)

print("--- Logistic Regression Baseline ---")
print(f"PRAUC:   {average_precision_score(y_test, lr_probs):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, lr_probs):.4f}")
print(classification_report(y_test, lr_preds))

In [ ]:
import optuna
import xgboost as xgb
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split

# ==========================================
# 1. PREPROCESS ONCE (Save Compute)
# ==========================================
print("Preprocessing data once before tuning...")

# Assume X_train, y_train are your raw data from before
# We do a secondary time-based split just for Optuna validation
X_opt_train, X_opt_val, y_opt_train, y_opt_val = train_test_split(
    X_train, y_train, test_size=0.2, shuffle=False
)

# Initialize just the preprocessing steps
preprocessor_pipeline = Pipeline([
    ('device_preprocessor', DevicePreprocessor(min_frequency=200)),
    ('general_preprocessor', DataPreprocessor(nan_threshold=0.01))
])

# Transform the datasets
X_train_processed = preprocessor_pipeline.fit_transform(X_opt_train, y_opt_train)
X_val_processed = preprocessor_pipeline.transform(X_opt_val)

scale_weight = (len(y_opt_train) - sum(y_opt_train)) / sum(y_opt_train)

# ==========================================
# 2. DEFINE THE OPTUNA OBJECTIVE
# ==========================================
def objective(trial):
    # 1. Define the hyperparameter search space
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        'scale_pos_weight': scale_weight,
        'tree_method': 'hist',
        'enable_categorical': True,
        'random_state': 42,
        'n_jobs': -1 # Use all CPU cores
    }

    # 2. Initialize and train the model
    model = xgb.XGBClassifier(**param)
    
    # We use early stopping so bad models abort quickly
    model.fit(
        X_train_processed, y_opt_train,
        eval_set=[(X_val_processed, y_opt_val)],
        verbose=False
    )

    # 3. Evaluate using the PR-AUC metric
    preds = model.predict_proba(X_val_processed)[:, 1]
    pr_auc = average_precision_score(y_opt_val, preds)
    
    return pr_auc

# ==========================================
# 3. RUN THE STUDY
# ==========================================
print("Starting Optuna optimization...")

# We want to MAXIMIZE the PR-AUC score
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30) # 30 trials is usually enough for a solid baseline

print("\n--- Optimization Finished ---")
print(f"Best PR-AUC Score: {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# ==========================================
# 4. BUILD THE FINAL PRODUCTION PIPELINE
# ==========================================
# Now you take the best parameters, put them back into your original pipeline, 
# and train it on the FULL training dataset.

best_xgb = xgb.XGBClassifier(
    **study.best_params, 
    scale_pos_weight=scale_weight,
    tree_method='hist',
    enable_categorical=True,
    random_state=42
)

final_production_pipeline = Pipeline([
    ('device_preprocessor', DevicePreprocessor(min_frequency=200)),
    ('general_preprocessor', DataPreprocessor(missing_threshold=0.01)),
    ('classifier', best_xgb)
])

# Train on the absolute full dataset and save to joblib
# final_production_pipeline.fit(X_train, y_train) 
# joblib.dump(final_production_pipeline, 'shopify_fraud_pipeline_optimized.joblib')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler

class AutoEncoder(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
        )

        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

    

In [ ]:
class AutoencoderFeatureExtractor(BaseEstimator, TransformerMixin):

    def __init__(self, epochs=20, batch_size=256, learning_rate=0.001):
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.scaler = StandardScaler()
        self.model = None

    def fit(self, X, y=None):
        """
        LEARNING PHASE: We only train the Autoencoder on NON-FRAUD data
        so it learns exactly what a "normal" transaction looks like.
        """

        X_fit = X.copy()

        if y is not None:
            # Convert y to a series if it's a numpy array to ensure index matching
            y_series = pd.Series(y, index=X_fit.index) if isinstance(y, np.ndarray) else y
            X_fit = X_fit[y_series == 0]
            print(f"Training AE on {len(X_fit)} legitimate transactions...")
            
        X_scaled = self.scaler.fit_transform(X_fit)
        
        X_tensor = torch.FloatTensor(X_scaled).to(self.device)
        dataset = torch.utils.data.TensorDataset(X_tensor, X_tensor)
        dataloader = torch.utils.data.DataLoader(dataset, batch_size=self.batch_size, shuffle=True)


        self.model = AutoEncoder(input_dim=X.shape[1]).to(self.device)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)

        print(f"Training Autoencoder on {self.device} for {self.epochs} epochs...")
        self.model.train()
        for epoch in range(self.epochs):
            total_loss = 0
            for batch_x, _ in dataloader:
                optimizer.zero_grad()
                outputs = self.model(batch_x)
                loss = criterion(outputs, batch_x)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            
            # Print progress every 5 epochs
            if (epoch + 1) % 5 == 0:
                print(f"Epoch {epoch+1}/{self.epochs} | Loss: {total_loss/len(dataloader):.4f}")

        return self
    
    def transform(self, X):
        """
        INFERENCE PHASE: Pass all data (train/test/live API) through the 
        trained model to calculate the Reconstruction Error.
        """
        X_copy = X.copy()
        
        X_scaled = self.scaler.transform(X_copy)
        X_tensor = torch.FloatTensor(X_scaled).to(self.device)
        
        self.model.eval()
        with torch.no_grad():
            reconstructed = self.model(X_tensor)
            
            # Calculate Mean Squared Error (MSE) per row
            mse = torch.mean((X_tensor - reconstructed) ** 2, dim=1).cpu().numpy()
            
        # Add the anomaly score as a brand new feature for XGBoost
        X_copy['ae_reconstruction_error'] = mse
        
        return X_copy

In [ ]:
class AutoencoderPipeline(BaseEstimator, TransformerMixin):
    def __init__(self, ae_extractor):
        self.ae_extractor = ae_extractor
        self.numeric_cols = None
        
    def fit(self, X, y=None):

        self.numeric_cols = X.select_dtypes(include=['int8', 'int16', 'int32', 'int64', 'float16', 'float32', 'float64']).columns
        # Train AE only on numeric data
        self.ae_extractor.fit(X[self.numeric_cols], y)
        return self
        
    def transform(self, X):
        X_out = X.copy()
        # Get the reconstruction error from the numeric data
        X_with_ae = self.ae_extractor.transform(X[self.numeric_cols])
        # Append just the new error column to our main dataframe
        X_out['ae_reconstruction_error'] = X_with_ae['ae_reconstruction_error']
        return X_out

In [ ]:
# Extract the trained XGBoost model from the pipeline
trained_xgb = shopify_pipeline.named_steps['classifier']

# Get the list of all features XGBoost sees
xgb_features = trained_xgb.feature_names_in_

if 'ae_reconstruction_error' in xgb_features:
    print("✅ SUCCESS! XGBoost sees the Autoencoder column.")
    
    # BONUS: Let's see how important the Autoencoder actually is!
    import pandas as pd
    
    # Extract importances
    importances = trained_xgb.feature_importances_
    
    # Create a dataframe and sort it
    importance_df = pd.DataFrame({
        'Feature': xgb_features,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)
    
    print("\nTop 10 Most Important Features:")
    print(importance_df.head(10))
    
    # Find exactly where the AE column ranks
    ae_rank = importance_df[importance_df['Feature'] == 'ae_reconstruction_error']
    print("\nAutoencoder Feature Rank:")
    print(ae_rank)
else:
    print("❌ FAILED. XGBoost did not receive the column.")

In [ ]:
# Print importance for your engineered features specifically
interpretable = [
    'TransactionAmt_log', 'TransactionAmt_decimal',
    'hour_of_day', 'day_of_week',
    'email_match', 'P_email_suffix', 'R_email_suffix',
    'OS', 'browser', 'screen_width', 'screen_height',
    'device_name', 'had_id',
    'ae_reconstruction_error'
]

imp_df = pd.DataFrame({
    'feature': trained_xgb.feature_names_in_,
    'importance': trained_xgb.feature_importances_
}).sort_values('importance', ascending=False)

print(imp_df[imp_df['feature'].isin(interpretable)])

In [ ]:
import matplotlib.pyplot as plt

# After transform, before XGBoost
ae_scores = X_train['ae_reconstruction_error']
fraud_scores  = ae_scores[y_train == 1]
normal_scores = ae_scores[y_train == 0]

plt.figure(figsize=(10, 4))
plt.hist(normal_scores, bins=100, alpha=0.5, label='Normal', density=True)
plt.hist(fraud_scores,  bins=100, alpha=0.5, label='Fraud',  density=True)
plt.xlim(0, np.percentile(ae_scores, 99))  # clip outliers for visibility
plt.legend()
plt.title("Reconstruction error distribution")
plt.show()

# Quick discrimination check
from sklearn.metrics import roc_auc_score
print(f"AE score AUC alone: {roc_auc_score(y_train, ae_scores):.4f}")
# If this is below 0.60, the autoencoder isn't learning useful signal

In [7]:
from sklearn.pipeline import Pipeline
data_pipeline = Pipeline([
    ('device_preprocessor', DevicePreprocessor(min_frequency=250)),
    ('general_preprocessor', DataPreprocessor(nan_threshold=0.01))
])

y = train_data['isFraud']
X = train_data.drop(['isFraud', 'TransactionID'], axis=1)

train_data = pd.DataFrame(data_pipeline.fit_transform(X, y))

df1 = pd.read_csv('../data/test_transaction.csv')
df2 = pd.read_csv('../data/test_identity.csv')

df_test = df1.merge(df2, on='TransactionID', how='left')
df_test = df_test.drop(['TransactionID'], axis=1)

df_test = pd.DataFrame(data_pipeline.transform(df_test))


ProductCD
card4
card6
P_emaildomain
R_emaildomain
M1
M2
M3
M4
M5
M6
M7
M8
M9
id_12
id_15
id_16
id_23
id_27
id_28
id_29
id_34
id_35
id_36
id_37
id_38
DeviceType
device_name
P_email_suffix
R_email_suffix
OS
version_OS
browser
version_browser
screen_width
screen_height
ProductCD
card4
card6
P_emaildomain
R_emaildomain
M1
M2
M3
M4
M5
M6
M7
M8
M9
id_12
id_15
id_16
id_23
id_27
id_28
id_29
id_34
id_35
id_36
id_37
id_38
DeviceType
device_name
P_email_suffix
R_email_suffix
OS
version_OS
browser
version_browser
screen_width
screen_height


In [12]:

df1 = pd.read_csv('../data/test_transaction.csv')
df2 = pd.read_csv('../data/test_identity.csv')

df_test = df1.merge(df2, on='TransactionID', how='left')
df_test = df_test.drop(['TransactionID'], axis=1)

#df_test = pd.DataFrame(data_pipeline.transform(df_test))


In [16]:
first_row = test_data.head(1)
first_row_processed = pd.DataFrame(data_pipeline.transform(first_row))

ProductCD
card4
card6
P_emaildomain
R_emaildomain
M1
M2
M3
M4
M5
M6
M7
M8
M9
id_12
id_15
id_16
id_23
id_27
id_28
id_29
id_34
id_35
id_36
id_37
id_38
DeviceType
device_name
P_email_suffix
R_email_suffix
OS
version_OS
browser
version_browser
screen_width
screen_height


In [27]:
first_row

,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,3663549,18403224,31.95,W,10409,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
cols = ['V228', 'V220', 'V167', 'V259', 'V192', 'V330', 'id_07', 'V199', 'V175', 'V337', 'V200', 'V258', 'V146', 'D13', 'id_03', 'V206', 'id_22', 'V138', 'V262', 'V334', 'V255', 'V324', 'V212', 'V139', 'V231', 'V338', 'V163', 'V183', 'V269', 'V149', 'V275', 'V166', 'V235', 'id_18', 'V240', 'V142', 'V267', 'V209', 'V173', 'id_24', 'V169', 'V328', 'V203', 'id_04', 'D12', 'id_19', 'id_25', 'D6', 'V161', 'V198', 'V158', 'V145', 'V238', 'V270', 'V230', 'V246', 'id_06', 'V254', 'V263', 'V253', 'id_09', 'V165', 'id_02', 'V186', 'id_11', 'V335', 'V245', 'D8', 'V160', 'V178', 'V182', 'V211', 'V339', 'V252', 'V272', 'V326', 'V164', 'V336', 'V264', 'V193', 'V215', 'V247', 'V237', 'V227', 'V331', 'V225', 'V226', 'V223', 'V217', 'id_17', 'V216', 'V180', 'V205', 'V250', 'V233', 'V241', 'V140', 'V332', 'D7', 'V174', 'V224', 'V188', 'V190', 'V184', 'V322', 'V327', 'V185', 'V144', 'V243', 'V232', 'V170', 'V148', 'dist2', 'V248', 'V276', 'V333', 'V323', 'V239', 'id_08', 'V195', 'V265', 'V156', 'V191', 'V147', 'D9', 'id_20', 'id_13', 'V171', 'V251', 'V177', 'V236', 'id_10', 'V234', 'D14', 'id_14', 'V229', 'id_21', 'V150', 'V189', 'id_32', 'V152', 'V271', 'V260', 'V278', 'V141', 'V268', 'V325', 'id_05', 'V207', 'V151', 'id_26', 'V266', 'V274', 'V244', 'V213', 'V256', 'V277', 'V168', 'V214', 'V196', 'V249', 'V179', 'V154', 'V210', 'id_01', 'V153', 'V221', 'V204', 'V172', 'V261', 'V201', 'V219', 'V143', 'V329', 'V202', 'V194', 'V273', 'V159', 'V208', 'V257', 'V222', 'V181', 'V187', 'V218', 'V197', 'V242', 'V162', 'V176', 'V155', 'V157']

In [ ]:
request = {
  "TransactionID": 3663549,
  "TransactionDT": 18403224,
  "TransactionAmt": 31.95,
  "ProductCD": "W",
  "card1": 10409,
  "card2": 111.0,
  "card3": 150.0,
  "card4": "visa",
  "card5": 226.0,
  "card6": "debit",
  "addr1": 170.0,
  "addr2": 87.0,
  "dist1": 1.0,
  "dist2": null,
  "P_emaildomain": "gmail.com",
  "R_emaildomain": null,
  "C1": 6.0,
  "C2": 6.0,
  "C3": 0.0,
  "C4": 0.0,
  "C5": 3.0,
  "C6": 4.0,
  "C7": 0.0,
  "C8": 0.0,
  "C9": 6.0,
  "C10": 0.0,
  "C11": 5.0,
  "C12": 1.0,
  "C13": 115.0,
  "C14": 6.0,
  "D1": 419.0,
  "D2": 419.0,
  "D3": 27.0,
  "D4": 398.0,
  "D5": 27.0,
  "D6": null,
  "D7": null,
  "D8": null,
  "D9": null,
  "D10": 418.0,
  "D11": 203.0,
  "D12": null,
  "D13": null,
  "D14": null,
  "D15": 409.0,
  "M1": "T",
  "M2": "T",
  "M3": "F",
  "M4": null,
  "M5": null,
  "M6": "F",
  "M7": "T",
  "M8": "T",
  "M9": "T",
  "V1": 1.0,
  "V2": 1.0,
  "V3": 1.0,
  "V4": 1.0,
  "V5": 1.0,
  "V6": 1.0,
  "V7": 1.0,
  "V8": 1.0,
  "V9": 1.0,
  "V10": 1.0,
  "V11": 1.0,
  "V12": 0.0,
  "V13": 0.0,
  "V14": 1.0,
  "V15": 0.0,
  "V16": 0.0,
  "V17": 0.0,
  "V18": 0.0,
  "V19": 0.0,
  "V20": 0.0,
  "V21": 0.0,
  "V22": 0.0,
  "V23": 1.0,
  "V24": 1.0,
  "V25": 0.0,
  "V26": 0.0,
  "V27": 0.0,
  "V28": 0.0,
  "V29": 0.0,
  "V30": 0.0,
  "V31": 0.0,
  "V32": 0.0,
  "V33": 0.0,
  "V34": 0.0,
  "V35": 1.0,
  "V36": 1.0,
  "V37": 1.0,
  "V38": 1.0,
  "V39": 0.0,
  "V40": 0.0,
  "V41": 1.0,
  "V42": 0.0,
  "V43": 0.0,
  "V44": 1.0,
  "V45": 1.0,
  "V46": 1.0,
  "V47": 1.0,
  "V48": 1.0,
  "V49": 1.0,
  "V50": 0.0,
  "V51": 0.0,
  "V52": 0.0,
  "V53": 0.0,
  "V54": 0.0,
  "V55": 1.0,
  "V56": 1.0,
  "V57": 0.0,
  "V58": 0.0,
  "V59": 0.0,
  "V60": 0.0,
  "V61": 0.0,
  "V62": 0.0,
  "V63": 0.0,
  "V64": 0.0,
  "V65": 1.0,
  "V66": 0.0,
  "V67": 0.0,
  "V68": 0.0,
  "V69": 0.0,
  "V70": 0.0,
  "V71": 0.0,
  "V72": 0.0,
  "V73": 0.0,
  "V74": 0.0,
  "V75": 0.0,
  "V76": 0.0,
  "V77": 1.0,
  "V78": 1.0,
  "V79": 0.0,
  "V80": 0.0,
  "V81": 0.0,
  "V82": 1.0,
  "V83": 1.0,
  "V84": 0.0,
  "V85": 0.0,
  "V86": 1.0,
  "V87": 1.0,
  "V88": 1.0,
  "V89": 0.0,
  "V90": 1.0,
  "V91": 1.0,
  "V92": 0.0,
  "V93": 0.0,
  "V94": 0.0,
  "V95": 0.0,
  "V96": 1.0,
  "V97": 0.0,
  "V98": 0.0,
  "V99": 1.0,
  "V100": 0.0,
  "V101": 0.0,
  "V102": 0.0,
  "V103": 0.0,
  "V104": 0.0,
  "V105": 0.0,
  "V106": 0.0,
  "V107": 1.0,
  "V108": 1.0,
  "V109": 1.0,
  "V110": 1.0,
  "V111": 1.0,
  "V112": 1.0,
  "V113": 1.0,
  "V114": 1.0,
  "V115": 1.0,
  "V116": 1.0,
  "V117": 1.0,
  "V118": 1.0,
  "V119": 1.0,
  "V120": 1.0,
  "V121": 1.0,
  "V122": 1.0,
  "V123": 1.0,
  "V124": 1.0,
  "V125": 1.0,
  "V126": 0.0,
  "V127": 47.95000076293945,
  "V128": 0.0,
  "V129": 0.0,
  "V130": 47.95000076293945,
  "V131": 0.0,
  "V132": 0.0,
  "V133": 0.0,
  "V134": 0.0,
  "V135": 0.0,
  "V136": 0.0,
  "V137": 0.0,
  "V138": null,
  "V139": null,
  "V140": null,
  "V141": null,
  "V142": null,
  "V143": null,
  "V144": null,
  "V145": null,
  "V146": null,
  "V147": null,
  "V148": null,
  "V149": null,
  "V150": null,
  "V151": null,
  "V152": null,
  "V153": null,
  "V154": null,
  "V155": null,
  "V156": null,
  "V157": null,
  "V158": null,
  "V159": null,
  "V160": null,
  "V161": null,
  "V162": null,
  "V163": null,
  "V164": null,
  "V165": null,
  "V166": null,
  "V167": null,
  "V168": null,
  "V169": null,
  "V170": null,
  "V171": null,
  "V172": null,
  "V173": null,
  "V174": null,
  "V175": null,
  "V176": null,
  "V177": null,
  "V178": null,
  "V179": null,
  "V180": null,
  "V181": null,
  "V182": null,
  "V183": null,
  "V184": null,
  "V185": null,
  "V186": null,
  "V187": null,
  "V188": null,
  "V189": null,
  "V190": null,
  "V191": null,
  "V192": null,
  "V193": null,
  "V194": null,
  "V195": null,
  "V196": null,
  "V197": null,
  "V198": null,
  "V199": null,
  "V200": null,
  "V201": null,
  "V202": null,
  "V203": null,
  "V204": null,
  "V205": null,
  "V206": null,
  "V207": null,
  "V208": null,
  "V209": null,
  "V210": null,
  "V211": null,
  "V212": null,
  "V213": null,
  "V214": null,
  "V215": null,
  "V216": null,
  "V217": null,
  "V218": null,
  "V219": null,
  "V220": null,
  "V221": null,
  "V222": null,
  "V223": null,
  "V224": null,
  "V225": null,
  "V226": null,
  "V227": null,
  "V228": null,
  "V229": null,
  "V230": null,
  "V231": null,
  "V232": null,
  "V233": null,
  "V234": null,
  "V235": null,
  "V236": null,
  "V237": null,
  "V238": null,
  "V239": null,
  "V240": null,
  "V241": null,
  "V242": null,
  "V243": null,
  "V244": null,
  "V245": null,
  "V246": null,
  "V247": null,
  "V248": null,
  "V249": null,
  "V250": null,
  "V251": null,
  "V252": null,
  "V253": null,
  "V254": null,
  "V255": null,
  "V256": null,
  "V257": null,
  "V258": null,
  "V259": null,
  "V260": null,
  "V261": null,
  "V262": null,
  "V263": null,
  "V264": null,
  "V265": null,
  "V266": null,
  "V267": null,
  "V268": null,
  "V269": null,
  "V270": null,
  "V271": null,
  "V272": null,
  "V273": null,
  "V274": null,
  "V275": null,
  "V276": null,
  "V277": null,
  "V278": null,
  "V279": 0.0,
  "V280": 0.0,
  "V281": 0.0,
  "V282": 0.0,
  "V283": 0.0,
  "V284": 0.0,
  "V285": 1.0,
  "V286": 0.0,
  "V287": 0.0,
  "V288": 0.0,
  "V289": 0.0,
  "V290": 1.0,
  "V291": 1.0,
  "V292": 1.0,
  "V293": 0.0,
  "V294": 0.0,
  "V295": 0.0,
  "V296": 0.0,
  "V297": 0.0,
  "V298": 0.0,
  "V299": 0.0,
  "V300": 0.0,
  "V301": 0.0,
  "V302": 0.0,
  "V303": 0.0,
  "V304": 0.0,
  "V305": 1.0,
  "V306": 0.0,
  "V307": 47.95000076293945,
  "V308": 0.0,
  "V309": 0.0,
  "V310": 47.95000076293945,
  "V311": 0.0,
  "V312": 0.0,
  "V313": 0.0,
  "V314": 0.0,
  "V315": 0.0,
  "V316": 0.0,
  "V317": 0.0,
  "V318": 0.0,
  "V319": 0.0,
  "V320": 0.0,
  "V321": 0.0,
  "V322": null,
  "V323": null,
  "V324": null,
  "V325": null,
  "V326": null,
  "V327": null,
  "V328": null,
  "V329": null,
  "V330": null,
  "V331": null,
  "V332": null,
  "V333": null,
  "V334": null,
  "V335": null,
  "V336": null,
  "V337": null,
  "V338": null,
  "V339": null,
  "id_01": null,
  "id_02": null,
  "id_03": null,
  "id_04": null,
  "id_05": null,
  "id_06": null,
  "id_07": null,
  "id_08": null,
  "id_09": null,
  "id_10": null,
  "id_11": null,
  "id_12": null,
  "id_13": null,
  "id_14": null,
  "id_15": null,
  "id_16": null,
  "id_17": null,
  "id_18": null,
  "id_19": null,
  "id_20": null,
  "id_21": null,
  "id_22": null,
  "id_23": null,
  "id_24": null,
  "id_25": null,
  "id_26": null,
  "id_27": null,
  "id_28": null,
  "id_29": null,
  "id_30": null,
  "id_31": null,
  "id_32": null,
  "id_33": null,
  "id_34": null,
  "id_35": null,
  "id_36": null,
  "id_37": null,
  "id_38": null,
  "DeviceType": null,
  "DeviceInfo": null
}



0   NaN
Name: V228, dtype: float64
0   NaN
Name: V220, dtype: float64
0   NaN
Name: V167, dtype: float64
0   NaN
Name: V259, dtype: float64
0   NaN
Name: V192, dtype: float64
0   NaN
Name: V330, dtype: float64
0   NaN
Name: id_07, dtype: float64
0   NaN
Name: V199, dtype: float64
0   NaN
Name: V175, dtype: float64
0   NaN
Name: V337, dtype: float64
0   NaN
Name: V200, dtype: float64
0   NaN
Name: V258, dtype: float64
0   NaN
Name: V146, dtype: float64
0   NaN
Name: D13, dtype: float64
0   NaN
Name: id_03, dtype: float64
0   NaN
Name: V206, dtype: float64
0   NaN
Name: id_22, dtype: float64
0   NaN
Name: V138, dtype: float64
0   NaN
Name: V262, dtype: float64
0   NaN
Name: V334, dtype: float64
0   NaN
Name: V255, dtype: float64
0   NaN
Name: V324, dtype: float64
0   NaN
Name: V212, dtype: float64
0   NaN
Name: V139, dtype: float64
0   NaN
Name: V231, dtype: float64
0   NaN
Name: V338, dtype: float64
0   NaN
Name: V163, dtype: float64
0   NaN
Name: V183, dtype: float64
0   NaN
Name: V269

In [18]:
import xgboost as xgb

model = xgb.XGBClassifier(
        n_estimators=700,
        max_depth=6,
        learning_rate=0.026,
        min_child_weight=8,
        subsample=0.957,
        colsample_bytree= 0.689,
        tree_method='hist',
        enable_categorical=True,
        random_state=42,
        eval_metric='aucpr', # Optimize for precision-recall area,
    )


model.fit(train_data, y)
model.predict(df_test)


array([0, 0, 0, ..., 0, 0, 0])

In [1]:
train_data

NameError: name 'train_data' is not defined